# 태영생막창 수익성 개선 프로젝트

- 실제 음식점 판매 데이터를 기반으로 수행된 분석 프로젝트
- 데이터 보안 이슈로 원본 데이터는 포함하지 않음
- 분석 로직 및 핵심 계산 과정 중심으로 재구성

## 분석 목표
- 매출 구조 파악 (파레토 분석)
- 수요 예측 기반 재고 전략 검토
- 공헌이익 기반 ABC 분석
- 손익분기점 분석 및 개선 전략 도출

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# 메뉴별 매출 데이터 (예시 구조)
df = pd.read_csv("../data/sales_data.csv")

menu_sales = df.groupby("menu")["sales"].sum().sort_values(ascending=False)

pareto_df = menu_sales.reset_index()
pareto_df["cum_ratio"] = pareto_df["sales"].cumsum() / pareto_df["sales"].sum()

display(pareto_df.head())

In [ ]:
fig, ax1 = plt.subplots()

ax1.bar(pareto_df["menu"], pareto_df["sales"])
ax2 = ax1.twinx()
ax2.plot(pareto_df["menu"], pareto_df["cum_ratio"], color="red", marker="o")

plt.title("파레토 분석 (메뉴별 매출)")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# 날짜 → 주차 변환
df["date"] = pd.to_datetime(df["date"])
df["week"] = df["date"].dt.isocalendar().week

weekly_sales = df.groupby(["menu", "week"])["quantity"].sum().reset_index()

display(weekly_sales.head())

In [ ]:
def moving_average(series, window=4):
    return series.rolling(window=window).mean()

sample = weekly_sales[weekly_sales["menu"] == "돼지생막창"]

sample = sample.sort_values("week")
sample["pred_ma"] = moving_average(sample["quantity"], 4)

In [ ]:
def mape(actual, pred):
    return np.mean(np.abs((actual - pred) / actual)) * 100

valid = sample.dropna()

mape_value = mape(valid["quantity"], valid["pred_ma"])
print(f"MAPE: {mape_value:.2f}%")

In [ ]:
# 예시
df["contribution"] = df["price"] - df["cost"]
df["total_contribution"] = df["contribution"] * df["quantity"]

contribution_df = (
    df.groupby("menu")["total_contribution"]
    .sum()
    .sort_values(ascending=False)
)

display(contribution_df)

In [ ]:
abc_df = contribution_df.reset_index()
abc_df["cum_ratio"] = abc_df["total_contribution"].cumsum() / abc_df["total_contribution"].sum()

def abc_class(x):
    if x <= 0.8:
        return "A"
    elif x <= 0.95:
        return "B"
    else:
        return "C"

abc_df["class"] = abc_df["cum_ratio"].apply(abc_class)

display(abc_df)

In [ ]:
# 예시 값
demand_mean = 90
demand_std = 20
lead_time = 0.3
z = 1.65  # 서비스 수준

safety_stock = z * demand_std * lead_time
rop = demand_mean * lead_time + safety_stock

print("안전재고:", round(safety_stock, 1))
print("ROP:", round(rop, 1))

In [ ]:
fixed_cost = 8936000
contribution_ratio = 0.61

bep = fixed_cost / contribution_ratio

print(f"손익분기점 매출: {bep:,.0f}원")

In [ ]:
before_sales = 14900000
after_sales = 16500000

growth = after_sales - before_sales

print(f"매출 증가: {growth:,.0f}원")

## 결론

- 매출은 소수 핵심 메뉴에 집중됨 (파레토 구조)
- 수요 예측은 제한적 성능 → 실무 적용 한계 확인
- 공헌이익 기반 ABC 분석이 재고 전략에 효과적
- ROP 및 안전재고 기반 운영 가능
- 이벤트는 전체 매출 증가에는 기여하지만 단일 품목 수익성은 감소